In [ ]:
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec

import seaborn as sns 

import gower
from sklearn.cluster import DBSCAN

import statsmodels.formula.api as smf 
from scipy.stats import spearmanr, sem, pearsonr, norm, chi2

from copy import deepcopy

import matplotlib
import matplotlib.pyplot as plt 
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator, PercentFormatter

from sklearn.metrics import mean_squared_error


import shap
import json
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import ElasticNet



# Utility functions and data processing 

## Loading SHAP data 

In [ ]:
OOS_PRED_COLUMNS_PUNISHPARAMS = ['CONFIG_playerCount',
 'CONFIG_numRounds',
 'CONFIG_showNRounds',
 'CONFIG_MPCR',
 'CONFIG_allOrNothing',
 'CONFIG_chat',
 'CONFIG_defaultContribProp',
 'CONFIG_rewardExists',
 'CONFIG_showOtherSummaries',
 'CONFIG_showPunishmentId',
 'CONFIG_punishmentCost',
 'CONFIG_punishmentTech']


DICT_COLUMN_LABELS = {'CONFIG_playerCount': 'Group Size',
 'CONFIG_numRounds': 'Interaction Length',
 'CONFIG_showNRounds': 'Horizon Knowledge',
 'CONFIG_MPCR': 'Return Rate (MPCR)',
 'CONFIG_allOrNothing': 'All-or-Nothing Choice',
 'CONFIG_chat': 'Peer Communication',
 'CONFIG_defaultContribProp': 'Opt-out Contributing',
 'CONFIG_rewardExists': 'Reward Mechanism',
 'CONFIG_showOtherSummaries': 'Peer Outcome Visibility',
 'CONFIG_showPunishmentId': 'Actor Anonymity',
 'CONFIG_punishmentCost': 'Punishment Cost',
 'CONFIG_punishmentTech': 'Punishment Impact',
 'control_itt_efficiency': 'Baseline Efficiency'}

df_paired_learn = pd.read_csv("./data/df_paired_learn.csv")



In [ ]:
class InteractionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Convert to numpy array if it's a pandas DataFrame
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()
        
        n_features = X.shape[1]
        interaction_terms = []
        for i in range(n_features):
            for j in range(i+1, n_features):
                interaction_terms.append(X[:, i] * X[:, j])
        
        # Combine original features and interaction terms
        result = np.column_stack([X] + interaction_terms)
        
        # Convert back to DataFrame if input was DataFrame
        if isinstance(X, pd.DataFrame):
            columns = list(X.columns) + [f'interact_{i}_{j}' for i in range(n_features) for j in range(i+1, n_features)]
            result = pd.DataFrame(result, columns=columns, index=X.index)
        
        return result

In [ ]:
prereg_hpo_params = json.load(open("./data/prereg_hpo_params_09202024.json", "r"))

elastic_prereg = Pipeline(steps=[("scaler", StandardScaler()),
                                 ("interactions", InteractionTransformer()),
                                 ("estimator", ElasticNet(alpha=prereg_hpo_params["ELASTIC"]["alpha"], 
                                                          l1_ratio=prereg_hpo_params["ELASTIC"]["l1_ratio"],
                                                          random_state=prereg_hpo_params["ELASTIC"]["random_seed"]))])

elastic_prereg.fit(X=df_paired_learn[OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]], y=df_paired_learn["treatment_itt_efficiency"])


enet_colnames = OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]
for i in range(len(OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"])):
    for j in range(i+1, len(OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"])):
        enet_colnames.append(f"{enet_colnames[i]}*{enet_colnames[j]}")
        
df_elastic_params = pd.DataFrame({"feature":enet_colnames, "coef":elastic_prereg[-1].coef_}).sort_values("coef", ascending=False)

In [ ]:
background_data = df_paired_learn[OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]].astype(float).values
masker = shap.maskers.Independent(background_data)

def model_wrapper(x):
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    return elastic_prereg.predict(x)

shap_explainer = shap.Explainer(
    model=model_wrapper,
    masker=masker,
    feature_names=OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]
)

shap_values = shap_explainer(df_paired_learn[OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]].astype(float).values)

# Figure 5 code

## Number of rounds

In [ ]:
df_elastic_params.query("feature.str.contains('CONFIG_numRounds') and abs(coef) > 0")

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(12,5))

# or you need more flexible customization
scatter1 = axes[0].scatter(
    shap_values[:, "CONFIG_numRounds"].data[np.where((shap_values[:, "CONFIG_chat"].data == 0) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 0))],
    shap_values[:, "CONFIG_numRounds"].values[np.where((shap_values[:, "CONFIG_chat"].data == 0) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 0))],
    c=shap_values[:, "CONFIG_showNRounds"].data[np.where((shap_values[:, "CONFIG_chat"].data == 0) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 0))],
    marker=".",
    cmap=plt.get_cmap("rainbow"),
    rasterized=True,
    zorder=5,
    label="Peer summaries disabled", s=100
)

axes[0].scatter(
    shap_values[:, "CONFIG_numRounds"].data[np.where((shap_values[:, "CONFIG_chat"].data == 0) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 1))],
    shap_values[:, "CONFIG_numRounds"].values[np.where((shap_values[:, "CONFIG_chat"].data == 0) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 1))],
    c=shap_values[:, "CONFIG_showNRounds"].data[np.where((shap_values[:, "CONFIG_chat"].data == 0) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 1))],
    marker="x",
    cmap=plt.get_cmap("rainbow"),
    rasterized=True,
    zorder=5,
    label="Peer summaries enabled", s=70
)

axes[0].set_title("Chat is disabled")
axes[0].set_xlabel("Round Count")
axes[0].set_ylabel("SHAP value")

scatter2 = axes[1].scatter(
    shap_values[:, "CONFIG_numRounds"].data[np.where((shap_values[:, "CONFIG_chat"].data == 1) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 0))],
    shap_values[:, "CONFIG_numRounds"].values[np.where((shap_values[:, "CONFIG_chat"].data == 1) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 0))],
    c=shap_values[:, "CONFIG_showNRounds"].data[np.where((shap_values[:, "CONFIG_chat"].data == 1) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 0))],
    marker=".",
    cmap=plt.get_cmap("rainbow"),
    rasterized=True,
    zorder=5,
    label="Peer summaries disabled", s=100
)

axes[1].scatter(
    shap_values[:, "CONFIG_numRounds"].data[np.where((shap_values[:, "CONFIG_chat"].data == 1) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 1))],
    shap_values[:, "CONFIG_numRounds"].values[np.where((shap_values[:, "CONFIG_chat"].data == 1) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 1))],
    c=shap_values[:, "CONFIG_showNRounds"].data[np.where((shap_values[:, "CONFIG_chat"].data == 1) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 1))],
    marker="x",
    cmap=plt.get_cmap("rainbow"),
    rasterized=True,
    zorder=5,
    label="Peer summaries enabled", s=70
)
axes[1].set_title("Chat is enabled")
axes[1].set_xlabel("Round Count")

plt.colorbar(scatter1, ax=axes[0], label='Show num rounds')
plt.colorbar(scatter2, ax=axes[1], label='Show num rounds')

plt.legend(markerscale=2)

## Opt in/out 

In [ ]:
df_elastic_params.query("feature.str.contains('CONFIG_default') and abs(coef) > 0")

In [ ]:
plt.scatter(
    shap_values[:, "CONFIG_defaultContribProp"].data[np.where(shap_values[:, "CONFIG_showOtherSummaries"].data == 0)],
    shap_values[:, "CONFIG_defaultContribProp"].values[np.where(shap_values[:, "CONFIG_showOtherSummaries"].data == 0)],
    c=shap_values[:, "CONFIG_allOrNothing"].data[np.where(shap_values[:, "CONFIG_showOtherSummaries"].data == 0)],
    marker="x",
    cmap=matplotlib.colors.ListedColormap(['blue', 'red']),
    rasterized=True,
    s=150,
    zorder=5,
    label="Peer Outcomes Hidden"
)

plt.scatter(
    shap_values[:, "CONFIG_defaultContribProp"].data[np.where(shap_values[:, "CONFIG_showOtherSummaries"].data == 1)],
    shap_values[:, "CONFIG_defaultContribProp"].values[np.where(shap_values[:, "CONFIG_showOtherSummaries"].data == 1)],
    c=shap_values[:, "CONFIG_allOrNothing"].data[np.where(shap_values[:, "CONFIG_showOtherSummaries"].data == 1)],
    marker="o",
    cmap=matplotlib.colors.ListedColormap(['blue', 'red']),
    rasterized=True,
    s=150,
    zorder=5,
    label="Peer Outcomes Shown"
)

plt.colorbar(label='All-or-Nothing Choice')
plt.xlabel("Opt-out Contributing")
plt.legend(loc="center", frameon=False)

In [ ]:
def shap_heatmap(shap_values):
    def get_shap_for_filter(CONFIG_showOtherSummaries, CONFIG_allOrNothing, CONFIG_defaultContribProp):
        return shap_values[:, "CONFIG_defaultContribProp"].values[np.where((shap_values[:, "CONFIG_showOtherSummaries"].data == CONFIG_showOtherSummaries)\
                                                            &(shap_values[:, "CONFIG_allOrNothing"].data == CONFIG_allOrNothing)\
                                                            &(shap_values[:, "CONFIG_defaultContribProp"].data == CONFIG_defaultContribProp))][0]
    
    heatmap_noSummaries = np.array([[get_shap_for_filter(0,0,0), get_shap_for_filter(0,0,1)],[get_shap_for_filter(0,1,0), get_shap_for_filter(0,1,1)]])
    print(heatmap_noSummaries)
    
    heatmap_withSummaries = np.array([[get_shap_for_filter(1,0,0), get_shap_for_filter(1,0,1)],[get_shap_for_filter(1,1,0), get_shap_for_filter(1,1,1)]])
    print(heatmap_withSummaries)
    
    # Find global min and max for consistent color scaling
    vmin = min(heatmap_noSummaries.min(), heatmap_withSummaries.min())
    vmax = max(heatmap_noSummaries.max(), heatmap_withSummaries.max())
    
    # Create figure with GridSpec
    fig = plt.figure(figsize=(12, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2])  # Second subplot slightly wider to accommodate colorbar
    
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharey=ax1)
    
    # Create heatmaps with shared vmin/vmax
    sns.heatmap(heatmap_noSummaries, annot=True, ax=ax1, 
                vmin=vmin, vmax=vmax, cbar=False)
    sns.heatmap(heatmap_withSummaries, annot=True, ax=ax2, 
                vmin=vmin, vmax=vmax, cbar_kws={'label': 'SHAP value'})
    
    ax1.set_yticks([0.5, 1.5], labels=["Continuous contribution", "All-or-nothing"])
    
    ax1.set_xticks([0.5, 1.5], labels=["Opt-in", "Opt-out"])
    ax2.set_xticks([0.5, 1.5], labels=["Opt-in", "Opt-out"])
    
    ax1.set_title("Peer Outcomes Not Shown")
    ax2.set_title("Peer Outcomes Shown")
    
    plt.tight_layout()

In [ ]:
shap_heatmap(shap_values)

## Punishment technology (technology = magnitude / cost)

In [ ]:
df_elastic_params.query("feature.str.contains('CONFIG_punishmentTech') and abs(coef) > 0")

In [ ]:
plt.scatter(
    shap_values[:, "CONFIG_punishmentTech"].data,
    shap_values[:, "CONFIG_punishmentTech"].values,
    c=shap_values[:, "CONFIG_allOrNothing"].data,
    marker="o",
    cmap=matplotlib.colors.ListedColormap(['blue', 'red']),
    rasterized=True,
    zorder=5,
)
plt.colorbar(label="All-or-nothing")
plt.xlabel("Punishment Tech")
plt.ylabel("SHAP value")

In [ ]:
#### here I start

In [ ]:
# [Cell 1] - Imports
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from matplotlib.ticker import MaxNLocator
import pandas as pd
import numpy as np
import shap
from sklearn.base import BaseEstimator, TransformerMixin
import json
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from matplotlib.lines import Line2D
import numpy as np


In [ ]:
OOS_PRED_COLUMNS_PUNISHPARAMS = ['CONFIG_playerCount',
 'CONFIG_numRounds',
 'CONFIG_showNRounds',
 'CONFIG_MPCR',
 'CONFIG_allOrNothing',
 'CONFIG_chat',
 'CONFIG_defaultContribProp',
 'CONFIG_rewardExists',
 'CONFIG_showOtherSummaries',
 'CONFIG_showPunishmentId',
 'CONFIG_punishmentCost',
 'CONFIG_punishmentTech']


DICT_COLUMN_LABELS = {'CONFIG_playerCount': 'Group Size',
 'CONFIG_numRounds': 'Interaction Length',
 'CONFIG_showNRounds': 'Horizon Knowledge',
 'CONFIG_MPCR': 'Return Rate (MPCR)',
 'CONFIG_allOrNothing': 'All-or-Nothing Choice',
 'CONFIG_chat': 'Peer Communication',
 'CONFIG_defaultContribProp': 'Opt-out Contributing',
 'CONFIG_rewardExists': 'Reward Mechanism',
 'CONFIG_showOtherSummaries': 'Peer Outcome Visibility',
 'CONFIG_showPunishmentId': 'Actor Anonymity',
 'CONFIG_punishmentCost': 'Punishment Cost',
 'CONFIG_punishmentTech': 'Punishment Impact',
 'control_itt_efficiency': 'Baseline Efficiency'}

df_paired_learn = pd.read_csv("./data/df_paired_learn.csv")



In [ ]:
df_paired_learn = pd.read_csv("./data/df_paired_learn.csv")

class InteractionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Convert to numpy array if it's a pandas DataFrame
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()
        
        n_features = X.shape[1]
        interaction_terms = []
        for i in range(n_features):
            for j in range(i+1, n_features):
                interaction_terms.append(X[:, i] * X[:, j])
        
        # Combine original features and interaction terms
        result = np.column_stack([X] + interaction_terms)
        
        # Convert back to DataFrame if input was DataFrame
        if isinstance(X, pd.DataFrame):
            columns = list(X.columns) + [f'interact_{i}_{j}' for i in range(n_features) for j in range(i+1, n_features)]
            result = pd.DataFrame(result, columns=columns, index=X.index)
        
        return result
    
    
prereg_hpo_params = json.load(open("./data/prereg_hpo_params_09202024.json", "r"))

elastic_prereg = Pipeline(steps=[("scaler", StandardScaler()),
                                 ("interactions", InteractionTransformer()),
                                 ("estimator", ElasticNet(alpha=prereg_hpo_params["ELASTIC"]["alpha"], 
                                                          l1_ratio=prereg_hpo_params["ELASTIC"]["l1_ratio"],
                                                          random_state=prereg_hpo_params["ELASTIC"]["random_seed"]))])

elastic_prereg.fit(X=df_paired_learn[OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]], y=df_paired_learn["treatment_itt_efficiency"])


enet_colnames = OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]
for i in range(len(OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"])):
    for j in range(i+1, len(OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"])):
        enet_colnames.append(f"{enet_colnames[i]}*{enet_colnames[j]}")
        
df_elastic_params = pd.DataFrame({"feature":enet_colnames, "coef":elastic_prereg[-1].coef_}).sort_values("coef", ascending=False)


background_data = df_paired_learn[OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]].astype(float).values
masker = shap.maskers.Independent(background_data)

def model_wrapper(x):
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    return elastic_prereg.predict(x)

shap_explainer = shap.Explainer(
    model=model_wrapper,
    masker=masker,
    feature_names=OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]
)

shap_values = shap_explainer(df_paired_learn[OOS_PRED_COLUMNS_PUNISHPARAMS+["control_itt_efficiency"]].astype(float).values)

In [ ]:
# [Cell 2] - Style Setup
plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['axes.facecolor'] = 'white'
matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['grid.alpha'] = 0.0


In [ ]:

# [Cell 3] - Colors and Markers Definition
# Nature-style colors
colors = {
    'peer_visible': '#E69F00',    # Orange
    'peer_hidden': '#0072B2'      # Blue
}

markers = {
    'peer_visible': 'o',          # Circle
    'peer_hidden': 's'           # Square
}

In [ ]:
# [Cell 4] - Create Figure
width_mm = 183  # Nature full width
height_mm = 90  # Adjusted for aspect ratio
width_inches = width_mm / 25.4
height_inches = height_mm / 25.4

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(width_inches, height_inches), 
                              sharey=True, dpi=300)

# [Cell 5] - Plot Data
# Left panel (Peer Communication disabled)
mask_no_comm_no_peer = (shap_values[:, "CONFIG_chat"].data == 0) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 0)
mask_no_comm_with_peer = (shap_values[:, "CONFIG_chat"].data == 0) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 1)

ax1.scatter(shap_values[:, "CONFIG_numRounds"].data[mask_no_comm_no_peer],
           shap_values[:, "CONFIG_numRounds"].values[mask_no_comm_no_peer],
           c=colors['peer_hidden'],
           marker=markers['peer_hidden'],
           s=40,
           alpha=0.7)

ax1.scatter(shap_values[:, "CONFIG_numRounds"].data[mask_no_comm_with_peer],
           shap_values[:, "CONFIG_numRounds"].values[mask_no_comm_with_peer],
           c=colors['peer_visible'],
           marker=markers['peer_visible'],
           s=40,
           alpha=0.7)

# Right panel (Peer Communication enabled)
mask_comm_no_peer = (shap_values[:, "CONFIG_chat"].data == 1) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 0)
mask_comm_with_peer = (shap_values[:, "CONFIG_chat"].data == 1) & (shap_values[:, "CONFIG_showOtherSummaries"].data == 1)

ax2.scatter(shap_values[:, "CONFIG_numRounds"].data[mask_comm_no_peer],
           shap_values[:, "CONFIG_numRounds"].values[mask_comm_no_peer],
           c=colors['peer_hidden'],
           marker=markers['peer_hidden'],
           s=40,
           alpha=0.7)

ax2.scatter(shap_values[:, "CONFIG_numRounds"].data[mask_comm_with_peer],
           shap_values[:, "CONFIG_numRounds"].values[mask_comm_with_peer],
           c=colors['peer_visible'],
           marker=markers['peer_visible'],
           s=40,
           alpha=0.7)

# [Cell 6] - Style and Labels
# Panel titles
ax1.set_title('Peer Communication Disabled', fontsize=10, pad=10)
ax2.set_title('Peer Communication Enabled', fontsize=10, pad=10)

# Axis labels
ax1.set_xlabel('Interaction Length', fontsize=9)
ax2.set_xlabel('Interaction Length', fontsize=9)
ax1.set_ylabel('SHAP Value', fontsize=9)

# Remove top and right spines
for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=8)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

# Create legend elements
legend_elements = [
    plt.Line2D([0], [0], marker=markers['peer_hidden'], color=colors['peer_hidden'],
               label='Peer Outcomes Hidden', markerfacecolor=colors['peer_hidden'],
               linestyle='None', markersize=6),
    plt.Line2D([0], [0], marker=markers['peer_visible'], color=colors['peer_visible'],
               label='Peer Outcomes Visible', markerfacecolor=colors['peer_visible'],
               linestyle='None', markersize=6)
]

# Add unified legend at the top
fig.legend(handles=legend_elements,
          loc='upper center',
          bbox_to_anchor=(0.5, 1.05),
          ncol=2,
          frameon=False,
          fontsize=8,
          handletextpad=0.5,
          handlelength=1.5,
          columnspacing=1.5)

# Adjust layout
plt.tight_layout()

# Save figure
plt.savefig('fig5_panel_a.pdf', 
            dpi=300, 
            bbox_inches='tight',
            metadata={'Creator': 'Matplotlib'})


In [ ]:
# Create figure
width_mm = 183  # Nature full width
height_mm = 100  # Slightly taller to accommodate larger panels
width_inches = width_mm / 25.4
height_inches = height_mm / 25.4

# Create figure with GridSpec for better control over panel sizes
fig = plt.figure(figsize=(width_inches, height_inches), dpi=300)
gs = plt.GridSpec(1, 3, width_ratios=[1.2, 1.2, 0.08])  # Larger panels, smaller colorbar
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
cax = fig.add_subplot(gs[2])  # Separate axis for colorbar

# [Previous data preparation code remains the same]

# Plot heatmaps
sns.heatmap(heatmap_peer_hidden, ax=ax1, cmap=cmap, 
            vmin=vmin, vmax=vmax, 
            annot=True, fmt='.3f', 
            cbar=False,
            square=True)

sns.heatmap(heatmap_peer_visible, ax=ax2, cmap=cmap, 
            vmin=vmin, vmax=vmax, 
            annot=True, fmt='.3f',
            cbar=True,
            cbar_ax=cax,  # Place colorbar in separate axis
            cbar_kws={'label': 'SHAP value'},
            square=True)

# Style the plots
ax1.set_title('Peer Outcomes Hidden', fontsize=10, pad=10)
ax2.set_title('Peer Outcomes Visible', fontsize=10, pad=10)

# Common x-axis label closer to panels with larger font
fig.text(0.5, 0.04, 'Contribution Framing', ha='center', fontsize=11)

# Style left panel (with y-tick labels)
ax1.set_xticks([0.5, 1.5])
ax1.set_xticklabels(['Opt-in', 'Opt-out'], fontsize=9)  # Increased from 8
ax1.set_yticks([0.5, 1.5])
ax1.set_yticklabels(['Variable\ncontribution', 'All or\nnothing'], 
                    fontsize=9, rotation=0)  # Increased from 8
ax1.tick_params(axis='both', which='major', labelsize=9)  # Increased from 8
for spine in ax1.spines.values():
    spine.set_linewidth(0.5)

# Style right panel (without y-tick labels)
ax2.set_xticks([0.5, 1.5])
ax2.set_xticklabels(['Opt-in', 'Opt-out'], fontsize=9)  # Increased from 8
ax2.set_yticks([0.5, 1.5])
ax2.set_yticklabels([])  # Remove y-tick labels
ax2.tick_params(axis='both', which='major', labelsize=9)  # Increased from 8
for spine in ax2.spines.values():
    spine.set_linewidth(0.5)

# Add y-label only to first panel with larger font
ax1.set_ylabel('Contribution Type', fontsize=11)  # Increased from 9
ax2.set_ylabel('')  # Remove y-label from second plot

# Force equal aspect ratio and size for both panels
ax1.set_aspect('equal')
ax2.set_aspect('equal')

# Adjust layout with tighter spacing
plt.subplots_adjust(left=0.1, right=0.9, bottom=0.12, top=0.9, wspace=0.2)

# Save figure
plt.savefig('fig5_panel_b.pdf', 
            dpi=300, 
            bbox_inches='tight',
            metadata={'Creator': 'Matplotlib'})


In [ ]:
# Create figure
width_mm = 183  # Nature full width
height_mm = 100  # Consistent with previous panels
width_inches = width_mm / 25.4
height_inches = height_mm / 25.4

# Define colors and markers (different from first panel)
colors = {
    'variable': '#7570B3',    # Purple
    'all_or_nothing': '#E7298A'  # Pink
}

markers = {
    'variable': 'D',          # Diamond
    'all_or_nothing': 'v'     # Triangle down
}

# Create figure
fig, ax = plt.subplots(figsize=(width_inches, height_inches), dpi=300)

# Create scatter plots for each condition
mask_variable = (shap_values[:, "CONFIG_allOrNothing"].data == 0)
mask_all_or_nothing = (shap_values[:, "CONFIG_allOrNothing"].data == 1)

# Plot Variable Contribution data
ax.scatter(
    shap_values[:, "CONFIG_punishmentTech"].data[mask_variable],
    shap_values[:, "CONFIG_punishmentTech"].values[mask_variable],
    color=colors['variable'],
    marker=markers['variable'],
    s=50,
    alpha=0.7,
    label='Variable contribution',
    zorder=5
)

# Plot All-or-Nothing data
ax.scatter(
    shap_values[:, "CONFIG_punishmentTech"].data[mask_all_or_nothing],
    shap_values[:, "CONFIG_punishmentTech"].values[mask_all_or_nothing],
    color=colors['all_or_nothing'],
    marker=markers['all_or_nothing'],
    s=50,
    alpha=0.7,
    label='All or nothing',
    zorder=5
)

# Style improvements
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)

# Labels
ax.set_xlabel('Punishment Technology (cost / impact)', fontsize=11)
ax.set_ylabel('SHAP Value', fontsize=11)

# Tick parameters
ax.tick_params(axis='both', which='major', labelsize=9, length=3, width=0.5)

# Add reference line at y=0
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)

# Add legend at the top
legend = fig.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, 1.05),
    ncol=2,
    frameon=False,
    fontsize=8,
    handletextpad=0.5,
    handlelength=1.5,
    columnspacing=1.5
)

# Adjust layout
plt.tight_layout()

# Save figure
plt.savefig('fig5_panel_c.pdf', 
            dpi=300, 
            bbox_inches='tight',
            metadata={'Creator': 'Matplotlib'})


In [ ]:
def create_heatmap(var1_col, var2_col, split_col, 
                  var1_name, var2_name, split_name,
                  var1_labels, var2_labels, split_labels):
    """
    var1_col: column for y-axis
    var2_col: column for x-axis
    split_col: column for panel split
    *_name: display names
    *_labels: display labels for values
    """
    
    width_mm = 183
    height_mm = 100
    width_inches = width_mm / 25.4
    height_inches = height_mm / 25.4

    fig = plt.figure(figsize=(width_inches, height_inches), dpi=300)
    gs = plt.GridSpec(1, 3, width_ratios=[1.2, 1.2, 0.08])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    cax = fig.add_subplot(gs[2])

    def get_shap_value(val1, val2, split_val):
        mask = ((shap_values[:, var1_col].data == val1) & 
                (shap_values[:, var2_col].data == val2) & 
                (shap_values[:, split_col].data == split_val))
        return shap_values[:, var2_col].values[mask].mean()

    # Create heatmaps for both split values
    heatmap_0 = np.array([[get_shap_value(i, j, 0) for j in [0,1]] 
                         for i in [0,1]])
    heatmap_1 = np.array([[get_shap_value(i, j, 1) for j in [0,1]] 
                         for i in [0,1]])

    vmin = min(heatmap_0.min(), heatmap_1.min())
    vmax = max(heatmap_0.max(), heatmap_1.max())
    cmap = sns.diverging_palette(220, 20, as_cmap=True)

    sns.heatmap(heatmap_0, ax=ax1, cmap=cmap, 
                vmin=vmin, vmax=vmax, 
                annot=True, fmt='.3f', 
                cbar=False,
                square=True)

    sns.heatmap(heatmap_1, ax=ax2, cmap=cmap, 
                vmin=vmin, vmax=vmax, 
                annot=True, fmt='.3f',
                cbar=True,
                cbar_ax=cax,
                cbar_kws={'label': 'SHAP value'},
                square=True)

    ax1.set_title(f'{split_name} = {split_labels[0]}', fontsize=10, pad=10)
    ax2.set_title(f'{split_name} = {split_labels[1]}', fontsize=10, pad=10)

    fig.text(0.5, 0.04, var2_name, ha='center', fontsize=11)

    # Style left panel
    ax1.set_xticks([0.5, 1.5])
    ax1.set_xticklabels(var2_labels, fontsize=9)
    ax1.set_yticks([0.5, 1.5])
    ax1.set_yticklabels(var1_labels, fontsize=9, rotation=0)
    ax1.tick_params(axis='both', which='major', labelsize=9)

    # Style right panel (no y-tick labels)
    ax2.set_xticks([0.5, 1.5])
    ax2.set_xticklabels(var2_labels, fontsize=9)
    ax2.set_yticks([0.5, 1.5])
    ax2.set_yticklabels([])
    ax2.tick_params(axis='both', which='major', labelsize=9)

    ax1.set_ylabel(var1_name, fontsize=11)
    ax2.set_ylabel('')

    ax1.set_aspect('equal')
    ax2.set_aspect('equal')

    plt.subplots_adjust(left=0.1, right=0.9, bottom=0.12, top=0.9, wspace=0.2)
    
    plt.savefig(f'heatmap_{var1_col}_{var2_col}_{split_col}.pdf', 
                dpi=300, bbox_inches='tight')
    #plt.close()

# Create all possible combinations
variables = {
    'CONFIG_defaultContribProp': ('Contribution Framing', ['Opt-in', 'Opt-out']),
    'CONFIG_allOrNothing': ('Contribution Type', ['Variable\ncontribution', 'All or\nnothing']),
    'CONFIG_showOtherSummaries': ('Peer Outcomes', ['Hidden', 'Visible'])
}

for var1 in variables:
    for var2 in variables:
        for split in variables:
            if var1 != var2 and var1 != split and var2 != split:
                create_heatmap(
                    var1, var2, split,
                    variables[var1][0], variables[var2][0], variables[split][0],
                    variables[var1][1], variables[var2][1], variables[split][1]
                )

In [ ]:

def get_cell_values():

    values = []

    for peer_outcomes in [0, 1]:  # Hidden, Visible

        for cont_type in [0, 1]:  # Variable, All-or-nothing

            for cont_frame in [0, 1]:  # Opt-in, Opt-out

                mask = ((shap_values[:, "CONFIG_showOtherSummaries"].data == peer_outcomes) & 

                       (shap_values[:, "CONFIG_allOrNothing"].data == cont_type) & 

                       (shap_values[:, "CONFIG_defaultContribProp"].data == cont_frame))

                

                shap_value = shap_values[:, "CONFIG_defaultContribProp"].values[mask].mean()

                

                values.append({

                    'Peer Outcomes': 'Visible' if peer_outcomes == 1 else 'Hidden',

                    'Contribution Type': 'All-or-nothing' if cont_type == 1 else 'Variable',

                    'Contribution Frame': 'Opt-out' if cont_frame == 1 else 'Opt-in',

                    'SHAP Value': shap_value

                })

    

    # Print formatted results

    for v in values:

        print(f"When Peer Outcomes is {v['Peer Outcomes']}, "

              f"Contribution Type is {v['Contribution Type']}, "

              f"and Contribution Frame is {v['Contribution Frame']}: "

              f"SHAP value = {v['SHAP Value']:.3f}")

    

    return values

cell_values = get_cell_values()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.lines import Line2D
import numpy as np

# Create figure
width_mm = 183
height_mm = 90
width_inches = width_mm / 25.4
height_inches = height_mm / 25.4

# Setup figure with two panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(width_inches, height_inches), dpi=300)

# Define distinct style elements
colors = {
    'Variable': '#7570B3',    # Purple
    'All-or-nothing': '#E7298A'  # Pink
}
markers = {
    'Variable': 'D',     # Diamond
    'All-or-nothing': 'v'  # Triangle down
}
linestyles = {
    'Variable': '-.',     # Dash-dot
    'All-or-nothing': ':'  # Dotted
}

def plot_interaction(ax, peer_outcome_val, title):
    for cont_type in [0, 1]:  # Variable vs All-or-nothing
        # Get SHAP values for this combination
        shap_values_opt_in = shap_values[:, "CONFIG_defaultContribProp"].values[
            (shap_values[:, "CONFIG_showOtherSummaries"].data == peer_outcome_val) & 
            (shap_values[:, "CONFIG_allOrNothing"].data == cont_type) &
            (shap_values[:, "CONFIG_defaultContribProp"].data == 0)
        ].mean()
        
        shap_values_opt_out = shap_values[:, "CONFIG_defaultContribProp"].values[
            (shap_values[:, "CONFIG_showOtherSummaries"].data == peer_outcome_val) & 
            (shap_values[:, "CONFIG_allOrNothing"].data == cont_type) &
            (shap_values[:, "CONFIG_defaultContribProp"].data == 1)
        ].mean()
        
        label = 'All-or-nothing' if cont_type == 1 else 'Variable'
        
        # Plot line
        ax.plot([0, 1], [shap_values_opt_in, shap_values_opt_out],
                color=colors[label],
                linestyle=linestyles[label],
                marker=markers[label],
                markersize=8,  # Slightly larger markers
                label=label,
                linewidth=2)   # Thicker lines for better visibility

    # Style plot
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Opt-in', 'Opt-out'], fontsize=9)
    ax.tick_params(axis='both', which='major', labelsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.set_title(title, fontsize=10, pad=10)
    
    # Add reference line at y=0
    ax.axhline(y=0, color='grey', linestyle='-', linewidth=0.5, alpha=0.3)

# Create panels
plot_interaction(ax1, 0, "Peer Outcomes Hidden")
plot_interaction(ax2, 1, "Peer Outcomes Visible")

# Labels
ax1.set_ylabel("SHAP Value", fontsize=11)
ax2.set_ylabel("")
fig.text(0.5, 0.01, "Contribution Framing", fontsize=11, ha='center')

# Set shared y-axis limits
y_min = min(ax1.get_ylim()[0], ax2.get_ylim()[0])
y_max = max(ax1.get_ylim()[1], ax2.get_ylim()[1])
ax1.set_ylim(y_min, y_max)
ax2.set_ylim(y_min, y_max)

# Add legend at top
handles = [
    Line2D([0], [0], color=colors['Variable'], linestyle=linestyles['Variable'],
           marker=markers['Variable'], label='Variable contribution',
           markersize=8, linewidth=2),
    Line2D([0], [0], color=colors['All-or-nothing'], linestyle=linestyles['All-or-nothing'],
           marker=markers['All-or-nothing'], label='All or nothing',
           markersize=8, linewidth=2)
]

fig.legend(handles=handles,
          loc='upper center', 
          bbox_to_anchor=(0.5, 1.05),
          ncol=2,
          frameon=False,
          fontsize=8,
          handletextpad=0.5,
          handlelength=1.5,
          columnspacing=1.5)

# Adjust layout
plt.tight_layout()

# Save figure
plt.savefig('interaction_plot.pdf', 
            dpi=300, 
            bbox_inches='tight',
            metadata={'Creator': 'Matplotlib'})


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.lines import Line2D
import numpy as np

# Style setup
plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['axes.facecolor'] = 'white'
matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['grid.alpha'] = 0.0

# Create figure with controlled spacing
width_mm = 183
height_mm = 220
width_inches = width_mm / 25.4
height_inches = height_mm / 25.4

fig = plt.figure(figsize=(width_inches, height_inches), dpi=300)
gs = plt.GridSpec(3, 2, height_ratios=[1, 1, 0.8], hspace=0.5, wspace=0.4)  # Increased hspace for clarity

# Create all axes
ax_game_length_1 = fig.add_subplot(gs[0, 0])
ax_game_length_2 = fig.add_subplot(gs[0, 1])
ax_interact_1 = fig.add_subplot(gs[1, 0])
ax_interact_2 = fig.add_subplot(gs[1, 1])
ax_punishment = fig.add_subplot(gs[2, :])

# Panel A & B setup: Game Length
colors_AB = {'peer_visible': '#E69F00', 'peer_hidden': '#0072B2'}
markers_AB = {'peer_visible': 'o', 'peer_hidden': 's'}

for idx, (ax, comm_enabled) in enumerate([(ax_game_length_1, 0), (ax_game_length_2, 1)]):
    for peer_visible, label in [(0, 'Peer Feedback Hidden'), (1, 'Peer Feedback Visible')]:
        mask = ((shap_values[:, "CONFIG_chat"].data == comm_enabled) &
                (shap_values[:, "CONFIG_showOtherSummaries"].data == peer_visible))
        
        ax.scatter(shap_values[:, "CONFIG_numRounds"].data[mask],
                   shap_values[:, "CONFIG_numRounds"].values[mask],
                   color=colors_AB['peer_visible' if peer_visible else 'peer_hidden'],
                   marker=markers_AB['peer_visible' if peer_visible else 'peer_hidden'],
                   s=40, alpha=0.7, label=label)
    ax.set_title(f'Peer Communication {"Enabled" if comm_enabled else "Disabled"}', fontsize=10, pad=10)
    ax.set_xlabel('Interaction Length', fontsize=9)

# Set shared y-axis limits and ticks for Panels A & B
y_min_AB, y_max_AB = min(ax_game_length_1.get_ylim()[0], ax_game_length_2.get_ylim()[0]), max(ax_game_length_1.get_ylim()[1], ax_game_length_2.get_ylim()[1])
ax_game_length_1.set_ylim(y_min_AB, y_max_AB)
ax_game_length_2.set_ylim(y_min_AB, y_max_AB)
ax_game_length_1.set_ylabel('SHAP Value', fontsize=9)

# Legend for Panels A & B, placed inside Panel A
handles_AB = [
    Line2D([0], [0], marker=markers_AB['peer_hidden'], color=colors_AB['peer_hidden'],
           label='Peer Feedback Hidden', markersize=8, linestyle='None'),
    Line2D([0], [0], marker=markers_AB['peer_visible'], color=colors_AB['peer_visible'],
           label='Peer Feedback Visible', markersize=8, linestyle='None')
]
ax_game_length_1.legend(handles=handles_AB, loc='upper right', frameon=False, fontsize=8)

# Panel C & D setup: Contribution Framing
colors_CD = {'Variable': '#7570B3', 'All-or-nothing': '#E7298A'}
markers_CD = {'Variable': 'D', 'All-or-nothing': 'v'}
linestyles_CD = {'Variable': '-.', 'All-or-nothing': ':'}

def plot_interaction(ax, peer_outcome_val, title):
    for cont_type, label in [(0, 'Variable contribution'), (1, 'All-or-nothing contribution')]:
        shap_values_opt_in = shap_values[:, "CONFIG_defaultContribProp"].values[
            (shap_values[:, "CONFIG_showOtherSummaries"].data == peer_outcome_val) & 
            (shap_values[:, "CONFIG_allOrNothing"].data == cont_type) &
            (shap_values[:, "CONFIG_defaultContribProp"].data == 0)
        ].mean()
        
        shap_values_opt_out = shap_values[:, "CONFIG_defaultContribProp"].values[
            (shap_values[:, "CONFIG_showOtherSummaries"].data == peer_outcome_val) & 
            (shap_values[:, "CONFIG_allOrNothing"].data == cont_type) &
            (shap_values[:, "CONFIG_defaultContribProp"].data == 1)
        ].mean()
        
        ax.plot([0, 1], [shap_values_opt_in, shap_values_opt_out],
                color=colors_CD[label.split()[0]],
                linestyle=linestyles_CD[label.split()[0]],
                marker=markers_CD[label.split()[0]],
                markersize=8,
                linewidth=2,
                label=label)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Opt-in', 'Opt-out'], fontsize=9)
    ax.set_title(title, fontsize=10, pad=10)
    ax.axhline(y=0, color='grey', linestyle='-', linewidth=0.5, alpha=0.3)

plot_interaction(ax_interact_1, 0, "Peer Outcomes Hidden")
plot_interaction(ax_interact_2, 1, "Peer Outcomes Visible")

# Set shared y-axis limits and ticks for Panels C & D
y_min_CD, y_max_CD = min(ax_interact_1.get_ylim()[0], ax_interact_2.get_ylim()[0]), max(ax_interact_1.get_ylim()[1], ax_interact_2.get_ylim()[1])
ax_interact_1.set_ylim(y_min_CD, y_max_CD)
ax_interact_2.set_ylim(y_min_CD, y_max_CD)
ax_interact_1.set_ylabel("SHAP Value", fontsize=9)

# Re-add x-axis labels for Panels C & D
ax_interact_1.set_xlabel("Contribution Framing", fontsize=9)
ax_interact_2.set_xlabel("Contribution Framing", fontsize=9)

# Legend for Panels C & D, placed inside Panel C
handles_CD = [
    Line2D([0], [0], color=colors_CD['Variable'], linestyle=linestyles_CD['Variable'],
           marker=markers_CD['Variable'], label='Variable contribution'),
    Line2D([0], [0], color=colors_CD['All-or-nothing'], linestyle=linestyles_CD['All-or-nothing'],
           marker=markers_CD['All-or-nothing'], label='All-or-nothing contribution')
]
ax_interact_1.legend(handles=handles_CD, loc='upper left', frameon=False, fontsize=8)

# Panel E setup: Punishment Technology
colors_E = {'Variable': '#7570B3', 'All-or-nothing': '#1B9E77'}
markers_E = {'Variable': 'D', 'All-or-nothing': 's'}

for cont_type, label in [(0, 'Variable contribution'), (1, 'All-or-nothing contribution')]:
    mask = (shap_values[:, "CONFIG_allOrNothing"].data == cont_type)
    ax_punishment.scatter(shap_values[:, "CONFIG_punishmentTech"].data[mask],
                          shap_values[:, "CONFIG_punishmentTech"].values[mask],
                          color=colors_E[label.split()[0]],
                          marker=markers_E[label.split()[0]],
                          s=40,
                          alpha=0.7,
                          label=label)
ax_punishment.set_xlabel('Punishment Technology (Impact/Cost)', fontsize=11)  # Updated x-axis label
ax_punishment.set_ylabel('SHAP Value', fontsize=11)

# Legend for Panel E, centered at the top of the panel
handles_E = [
    Line2D([0], [0], marker=markers_E['Variable'], color=colors_E['Variable'], 
           label='Variable contribution', markersize=8, linestyle='None'),
    Line2D([0], [0], marker=markers_E['All-or-nothing'], color=colors_E['All-or-nothing'],
           label='All-or-nothing contribution', markersize=8, linestyle='None')
]
ax_punishment.legend(handles=handles_E, loc='upper center', bbox_to_anchor=(0.5, 1.05), frameon=False, fontsize=8, ncol=2)

# Add panel labels and style all axes
for idx, ax in enumerate([ax_game_length_1, ax_game_length_2, ax_interact_1, ax_interact_2, ax_punishment]):
    ax.text(-0.1, 1.05, chr(65 + idx), transform=ax.transAxes, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Apply tight_layout for final adjustments
plt.tight_layout()
plt.savefig('fig5_complete_corrected.pdf', dpi=300, bbox_inches='tight')


In [ ]:
import numpy as np

# Simulate data (only needed if you don't already have `shap_values_df`)
np.random.seed(42)
num_samples = 100
data = {
    "CONFIG_chat": np.random.choice([0, 1], num_samples),                   # Communication Enabled/Disabled
    "CONFIG_showOtherSummaries": np.random.choice([0, 1], num_samples),     # Peer Feedback Visible/Hidden
    "CONFIG_numRounds": np.random.randint(5, 30, num_samples),              # Interaction Length
    "SHAP_Value": np.random.uniform(-0.02, 0.02, num_samples)               # SHAP Values
}
shap_values_df = pd.DataFrame(data)

# Define labels for configuration values
communication_labels = {0: "Disabled", 1: "Enabled"}
feedback_labels = {0: "Hidden", 1: "Visible"}

# Panel A: Peer Communication Disabled
print("Panel A - Peer Communication Disabled")
for idx, row in shap_values_df[shap_values_df["CONFIG_chat"] == 0].iterrows():
    comm = communication_labels[row["CONFIG_chat"]]
    feedback = feedback_labels[row["CONFIG_showOtherSummaries"]]
    rounds = row["CONFIG_numRounds"]
    shap_val = row["SHAP_Value"]
    print(f"Interaction Length: {rounds}, Peer Feedback: {feedback}, SHAP Value: {shap_val:.4f}")

# Panel B: Peer Communication Enabled
print("\nPanel B - Peer Communication Enabled")
for idx, row in shap_values_df[shap_values_df["CONFIG_chat"] == 1].iterrows():
    comm = communication_labels[row["CONFIG_chat"]]
    feedback = feedback_labels[row["CONFIG_showOtherSummaries"]]
    rounds = row["CONFIG_numRounds"]
    shap_val = row["SHAP_Value"]
    print(f"Interaction Length: {rounds}, Peer Feedback: {feedback}, SHAP Value: {shap_val:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.lines import Line2D
import numpy as np

# Style setup
plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['axes.facecolor'] = 'white'
matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['grid.alpha'] = 0.0

# Create figure with adjusted spacing (no Panel E)
width_mm = 183
height_mm = 160  # Adjust height to fit only 2 rows of panels
width_inches = width_mm / 25.4
height_inches = height_mm / 25.4

fig = plt.figure(figsize=(width_inches, height_inches), dpi=300)
gs = plt.GridSpec(2, 2, height_ratios=[1, 1], hspace=0.5, wspace=0.4)  # Only 2 rows for Panels A, B, C, and D

# Create all axes
ax_game_length_1 = fig.add_subplot(gs[0, 0])
ax_game_length_2 = fig.add_subplot(gs[0, 1])
ax_interact_1 = fig.add_subplot(gs[1, 0])
ax_interact_2 = fig.add_subplot(gs[1, 1])

# Panel A & B setup: Game Length
colors_AB = {'peer_visible': '#E69F00', 'peer_hidden': '#0072B2'}
markers_AB = {'peer_visible': 'o', 'peer_hidden': 's'}

for idx, (ax, comm_enabled) in enumerate([(ax_game_length_1, 0), (ax_game_length_2, 1)]):
    for peer_visible, label in [(0, 'Peer Feedback Hidden'), (1, 'Peer Feedback Visible')]:
        mask = ((shap_values[:, "CONFIG_chat"].data == comm_enabled) &
                (shap_values[:, "CONFIG_showOtherSummaries"].data == peer_visible))
        
        ax.scatter(shap_values[:, "CONFIG_numRounds"].data[mask],
                   shap_values[:, "CONFIG_numRounds"].values[mask],
                   color=colors_AB['peer_visible' if peer_visible else 'peer_hidden'],
                   marker=markers_AB['peer_visible' if peer_visible else 'peer_hidden'],
                   s=40, alpha=0.7, label=label)
    ax.set_title(f'Peer Communication {"Enabled" if comm_enabled else "Disabled"}', fontsize=10, pad=10)
    ax.set_xlabel('Game Length', fontsize=9)

# Set shared y-axis limits and ticks for Panels A & B
y_min_AB, y_max_AB = min(ax_game_length_1.get_ylim()[0], ax_game_length_2.get_ylim()[0]), max(ax_game_length_1.get_ylim()[1], ax_game_length_2.get_ylim()[1])
ax_game_length_1.set_ylim(y_min_AB, y_max_AB)
ax_game_length_2.set_ylim(y_min_AB, y_max_AB)
ax_game_length_1.set_ylabel('SHAP Value', fontsize=9)

# Legend for Panels A & B, placed inside Panel A
handles_AB = [
    Line2D([0], [0], marker=markers_AB['peer_hidden'], color=colors_AB['peer_hidden'],
           label='Peer Outcomes Hidden', markersize=8, linestyle='None'),
    Line2D([0], [0], marker=markers_AB['peer_visible'], color=colors_AB['peer_visible'],
           label='Peer Outcomes Visible', markersize=8, linestyle='None')
]
ax_game_length_1.legend(handles=handles_AB, loc='upper right', frameon=False, fontsize=8)

# Panel C & D setup: Contribution Framing
colors_CD = {'Variable': '#7570B3', 'All-or-nothing': '#E7298A'}
markers_CD = {'Variable': 'D', 'All-or-nothing': 'v'}
linestyles_CD = {'Variable': '-.', 'All-or-nothing': ':'}

def plot_interaction(ax, peer_outcome_val, title):
    for cont_type, label in [(0, 'Variable contribution'), (1, 'All-or-nothing contribution')]:
        shap_values_opt_in = shap_values[:, "CONFIG_defaultContribProp"].values[
            (shap_values[:, "CONFIG_showOtherSummaries"].data == peer_outcome_val) & 
            (shap_values[:, "CONFIG_allOrNothing"].data == cont_type) &
            (shap_values[:, "CONFIG_defaultContribProp"].data == 0)
        ].mean()
        
        shap_values_opt_out = shap_values[:, "CONFIG_defaultContribProp"].values[
            (shap_values[:, "CONFIG_showOtherSummaries"].data == peer_outcome_val) & 
            (shap_values[:, "CONFIG_allOrNothing"].data == cont_type) &
            (shap_values[:, "CONFIG_defaultContribProp"].data == 1)
        ].mean()
        
        ax.plot([0, 1], [shap_values_opt_in, shap_values_opt_out],
                color=colors_CD[label.split()[0]],
                linestyle=linestyles_CD[label.split()[0]],
                marker=markers_CD[label.split()[0]],
                markersize=8,
                linewidth=2,
                label=label)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Opt-in', 'Opt-out'], fontsize=9)
    ax.set_title(title, fontsize=10, pad=10)
    ax.axhline(y=0, color='grey', linestyle='-', linewidth=0.5, alpha=0.3)

plot_interaction(ax_interact_1, 0, "Peer Outcomes Hidden")
plot_interaction(ax_interact_2, 1, "Peer Outcomes Visible")

# Set shared y-axis limits and ticks for Panels C & D
y_min_CD, y_max_CD = min(ax_interact_1.get_ylim()[0], ax_interact_2.get_ylim()[0]), max(ax_interact_1.get_ylim()[1], ax_interact_2.get_ylim()[1])
ax_interact_1.set_ylim(y_min_CD, y_max_CD)
ax_interact_2.set_ylim(y_min_CD, y_max_CD)
ax_interact_1.set_ylabel("SHAP Value", fontsize=9)

# Add x-axis labels for Panels C & D
ax_interact_1.set_xlabel("Contribution Framing", fontsize=9)
ax_interact_2.set_xlabel("Contribution Framing", fontsize=9)

# Legend for Panels C & D, placed inside Panel C
handles_CD = [
    Line2D([0], [0], color=colors_CD['Variable'], linestyle=linestyles_CD['Variable'],
           marker=markers_CD['Variable'], label='Variable contribution'),
    Line2D([0], [0], color=colors_CD['All-or-nothing'], linestyle=linestyles_CD['All-or-nothing'],
           marker=markers_CD['All-or-nothing'], label='All-or-nothing contribution')
]
ax_interact_1.legend(handles=handles_CD, loc='upper left', frameon=False, fontsize=8)

# Add panel labels and style all axes
for idx, ax in enumerate([ax_game_length_1, ax_game_length_2, ax_interact_1, ax_interact_2]):
    ax.text(-0.1, 1.05, chr(65 + idx), transform=ax.transAxes, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Apply tight_layout for final adjustments
plt.tight_layout()
plt.savefig('fig5_complete_corrected_no_panel_E.png', dpi=300, bbox_inches='tight')
